# 04 · M1 微调 + 四组退化扫描（一体化 · 阶段 B）

**一次 Run All 产出阶段 B 全部结果**：
1. M1 微调：EuroSAT 分类（Prithvi-EO-1.0，5 epoch，T4 约 18 分钟）
2. 四组退化扫描：云遮挡 0–70% / GSD 1.0–8.0× / SNR 30→6 dB / MTF σ 0.5–4.0
3. 四份 CSV + 四联响应曲线 + H1/H2/H4 自动分析

**前置**：右侧 + Add Input 添加**整个 eo-degrade 仓库**（Dataset 类型，含 `degrade/` 与 `scripts/`）；
**Settings：Accelerator = GPU，Internet = ON**。

**总耗时**：约 35–45 分钟（下载 5min + 训练 18min + 四组扫描 10–20min）。

In [ ]:
# 1) 环境准备
# numpy<2 必须先装（terratorch/albumentations 与 numpy 2.x 不兼容）
!pip install "numpy<2" -q 2>&1 | tail -1
!pip install -q terratorch torchgeo 2>&1 | tail -1
import torch
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('GPU 可用显存(GB):', round(torch.cuda.get_device_properties(0).total_memory/1e9,1) if torch.cuda.is_available() else 0)

In [ ]:
# 2) 挂载 eo-degrade 仓库（degrade 库 + scripts 扫描引擎）
import sys, os, glob
candidates = []
for root in ['/kaggle/input', '.']:
    for hit in glob.glob(os.path.join(root, '**', 'scripts', 'scan_degradation.py'), recursive=True):
        candidates.append(os.path.dirname(os.path.dirname(hit)))
if candidates:
    repo = candidates[0]
    sys.path.insert(0, repo)
    print('找到 eo-degrade 仓库:', repo)
else:
    raise FileNotFoundError('未找到 scripts/scan_degradation.py。请添加整个 eo-degrade 仓库（Dataset 类型）。')

from degrade.cloud import add_cloud_mask
from degrade.gsd import degrade_gsd
from degrade.snr import add_poisson_gaussian
from degrade.mtf import degrade_mtf
from scripts.scan_degradation import (
    gsd_scan_points, snr_scan_points_for, mtf_scan_points, mtf_kernel_size,
    cloud_scan_points, eval_degrade,
)
print('退化库与扫描引擎加载成功')

In [ ]:
# 3) 下载 EuroSAT（TorchGeo 自动从 Zenodo 下载，约 90MB；数据目录由 TorchGeo 自行管理）
import os
from torchgeo.datasets import EuroSAT

os.makedirs('data', exist_ok=True)
ds = EuroSAT(root='data', download=True)
print(f'EuroSAT 就绪，共 {len(ds)} 张（10 类）')

In [ ]:
# 4) M1 微调（与 01 相同的配置：Prithvi-EO-1.0 + 5 epoch）
yaml_content = '''
model:
  class_path: terratorch.tasks.ClassificationTask
  init_args:
    model_args:
      backbone: prithvi_eo_v1_100
      backbone_pretrained: true
      num_frames: 1
      bands:
        - BLUE
        - GREEN
        - RED
      num_classes: 10
      head_dropout: 0.1
    model_factory: EncoderDecoderFactory
    loss: ce
    lr: 1.0e-4

data:
  class_path: terratorch.datamodules.EuroSATDataModule
  init_args:
    root: data
    batch_size: 32
    num_workers: 2
    bands:
      - BLUE
      - GREEN
      - RED
    train_transform:
      - class_path: albumentations.Resize
        init_args: {height: 224, width: 224}
    val_transform:
      - class_path: albumentations.Resize
        init_args: {height: 224, width: 224}
    test_transform:
      - class_path: albumentations.Resize
        init_args: {height: 224, width: 224}

trainer:
  max_epochs: 5
  accelerator: auto
  devices: 1
  log_every_n_steps: 10
  default_root_dir: ./m1_run
'''
with open('eurosat_prithvi.yaml', 'w') as f:
    f.write(yaml_content)
print('训练配置已写入 eurosat_prithvi.yaml（权重将保存到 ./m1_run/lightning_logs）')

import subprocess
r = subprocess.run(['terratorch', 'fit', '--config', 'eurosat_prithvi.yaml'],
                   capture_output=True, text=True)
print('=== 训练 STDOUT 尾部 ===')
print(r.stdout[-2500:])
print('退出码:', r.returncode)

In [ ]:
# 5) 解析 M1 指标 + 从训练产物加载模型
import re
stdout = r.stdout
metrics = {}
for key in ['val_miou', 'val_accuracy', 'val_loss', 'test_accuracy', 'test_loss']:
    m = re.findall(rf'{key}\s*=\s*([0-9.]+)', stdout)
    if m:
        metrics[key] = float(m[-1])
print('=== M1 训练指标 ===')
for k, v in metrics.items():
    print(f'{k}: {v:.4f}')

from terratorch.tasks import ClassificationTask
import glob
# checkpoint 可能存于不同路径（m1_run/lightning_logs 或其他），全盘查找兜底
ckpts = sorted(set(
    glob.glob('m1_run/**/*.ckpt', recursive=True)
    + glob.glob('lightning_logs/**/*.ckpt', recursive=True)
    + glob.glob('**/*.ckpt', recursive=True)))
print('找到 checkpoint:', ckpts)
if not ckpts:
    import os
    for d in ['m1_run', 'lightning_logs']:
        if os.path.exists(d):
            print(f'-- {d} 目录结构 --')
            for root, dirs, files in os.walk(d):
                print(root, dirs[:5], files[:5])
    raise FileNotFoundError('未找到 checkpoint，请把上方目录结构截图发我')
model = ClassificationTask.load_from_checkpoint(ckpts[0], map_location='cpu')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
model.eval()
print('模型已加载（微调后）:', device)

In [ ]:
# 6) 测试集加载 + 值域诊断 + 数据均值（SNR 反解用）
from terratorch.datamodules import EuroSATDataModule

dm = EuroSATDataModule(
    root='data', batch_size=32, num_workers=2,
    bands=['BLUE', 'GREEN', 'RED'],
    test_transform=[{'class_path': 'albumentations.Resize', 'init_args': {'height': 224, 'width': 224}}],
)
dm.setup('test')
loader = dm.test_dataloader()
print('测试集 batch 数:', len(loader))

xb = next(iter(loader))['image']
GLOBAL_MAX = xb.max().item()
print('image 范围: min=%.3f max=%.3f' % (xb.min().item(), GLOBAL_MAX))

sum_im, n_pix = 0.0, 0
with torch.no_grad():
    for bch in loader:
        im = bch['image']
        if GLOBAL_MAX > 1.5:
            im = im / 255.0
        sum_im += im.sum().item()
        n_pix += im.numel()
X_MEAN = sum_im / n_pix
print('数据均值 X_MEAN = %.4f' % X_MEAN)

In [ ]:
# 7) 干净基线（对照点）
SEED = 0
clean_acc, clean_f1 = eval_degrade(model, loader, lambda im: im)
print(f'干净基线: acc={clean_acc:.4f} f1={clean_f1:.4f}')

In [ ]:
# 8) 云遮挡扫描（0–70%）→ H4
cloud_results = []
print('=== 云遮挡扫描 ===')
for frac in cloud_scan_points():
    acc, f1 = eval_degrade(model, loader, lambda im, f=frac: add_cloud_mask(im, f, seed=SEED)[0])
    cloud_results.append({'degrade': frac, 'accuracy': round(float(acc),4), 'f1': round(float(f1),4)})
    print(f'云 {frac:.0%} → acc {acc:.4f} f1 {f1:.4f}')

In [ ]:
# 9) GSD 扫描（1.0–8.0×）→ H1
gsd_results = []
print('=== GSD 扫描 ===')
for s in gsd_scan_points():
    acc, f1 = eval_degrade(model, loader, lambda im, s=s: degrade_gsd(im, s))
    gsd_results.append({'degrade': s, 'accuracy': round(float(acc),4), 'f1': round(float(f1),4)})
    print(f'GSD {s:.1f}× → acc {acc:.4f} f1 {f1:.4f}')

In [ ]:
# 10) SNR 扫描（30→6 dB）→ H2
snr_points = snr_scan_points_for(X_MEAN)
snr_results = []
print('=== SNR 扫描（目标 dB → a 反解）===')
for pt in snr_points:
    acc, f1 = eval_degrade(model, loader,
        lambda im, pt=pt: add_poisson_gaussian(im, a=pt['a'], b=pt['b'], seed=SEED))
    snr_results.append({'degrade': pt['db'], 'accuracy': round(float(acc),4), 'f1': round(float(f1),4)})
    print(f"SNR {pt['db']:.0f}dB (a={pt['a']:.5f}) → acc {acc:.4f} f1 {f1:.4f}")

In [ ]:
# 11) MTF 扫描（σ 0.5–4.0，核自适应）→ H3（EuroSAT 单数据集部分）
mtf_results = []
print('=== MTF 扫描 ===')
for s in mtf_scan_points():
    ks = mtf_kernel_size(s)
    acc, f1 = eval_degrade(model, loader, lambda im, s=s, ks=ks: degrade_mtf(im, s, kernel_size=ks))
    mtf_results.append({'degrade': s, 'accuracy': round(float(acc),4), 'f1': round(float(f1),4)})
    print(f'MTF σ={s:.1f} (kernel {ks}) → acc {acc:.4f} f1 {f1:.4f}')

In [ ]:
# 12) 保存四份 CSV
import csv
os.makedirs('/kaggle/working/results', exist_ok=True)
def save_csv(name, rows):
    p = f'/kaggle/working/results/{name}'
    with open(p, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=['degrade', 'accuracy', 'f1'])
        w.writeheader(); w.writerows(rows)
    print('已保存:', p)
save_csv('cloud_scan.csv', cloud_results)
save_csv('gsd_scan.csv', gsd_results)
save_csv('snr_scan.csv', snr_results)
save_csv('mtf_scan.csv', mtf_results)

In [ ]:
# 13) 四联响应曲线
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
def plot(ax, rows, xlab, title):
    xs = [r['degrade'] for r in rows]; ys = [r['accuracy']*100 for r in rows]
    ax.plot(xs, ys, 'o-', color='#2B5FB8', linewidth=2)
    ax.axhline(clean_acc*100, color='#6B7280', linestyle=':', linewidth=1, label=f'clean {clean_acc*100:.1f}%')
    ax.set_xlabel(xlab); ax.set_ylabel('精度 (%)'); ax.set_title(title)
    ax.grid(alpha=0.3); ax.legend(fontsize=9)
plot(axes[0], cloud_results, '云遮挡率', 'H4 云遮挡响应')
plot(axes[1], gsd_results, 'GSD (×)', 'H1 GSD 响应')
plot(axes[2], snr_results, 'SNR (dB)', 'H2 SNR 响应')
plot(axes[3], mtf_results, 'MTF σ', 'H3 MTF 响应 (EuroSAT)')
plt.tight_layout()
png = '/kaggle/working/results/m2_four_curves.png'
plt.savefig(png, dpi=150, bbox_inches='tight')
plt.show()
print('已保存:', png)

In [ ]:
# 14) H1/H2/H4 自动分析（诚实版：只对当前数据下结论）
def sweet_zone(rows, min_improve=0.01):
    """甜区：从该点起再提升参数，精度增益 <1% 视为收益递减。"""
    for i in range(len(rows)-1, 0, -1):
        if rows[i]['accuracy'] - rows[i-1]['accuracy'] >= min_improve:
            return rows[i]['degrade'], rows[i-1]['degrade']
    return None, None

def breakpoint(rows, min_drop=0.02):
    for i in range(1, len(rows)):
        if rows[i-1]['accuracy'] - rows[i]['accuracy'] >= min_drop:
            return rows[i]['degrade'], rows[i-1]['accuracy'] - rows[i]['accuracy']
    return None, None

def failure(rows, keep=0.7):
    base = rows[0]['accuracy']
    for r in rows:
        if r['accuracy'] < base * keep:
            return r['degrade'], r['accuracy']
    return None, None

print('=== 自动分析 ===')
print(f'干净基线: {clean_acc*100:.2f}%')

sg, _ = sweet_zone(gsd_results)
print(f'[H1 GSD] 甜区: 分辨率提升至 {sg}× 附近后收益递减' if sg else '[H1] 未检出甜区')

ss, _ = sweet_zone(snr_results)
print(f'[H2 SNR] 饱和点: SNR 超过 {ss}dB 后精度不再提升' if ss else '[H2] 未检出饱和点')

bf, drop = breakpoint(cloud_results)
ff, fa = failure(cloud_results)
if bf is not None:
    print(f'[H4 云] 加速下降拐点: 遮挡率 {bf:.0%}（该步降 {drop:.3f}）')
else:
    print('[H4 云] 未检出加速拐点（平滑下降或阴性）')
if ff is not None:
    print(f'[H4 云] 失效点: 遮挡率 {ff:.0%}（精度 {fa:.3f} < 干净×70%）')
else:
    print('[H4 云] 全程未跌破干净精度的 70%')

print()
print('注: H3 的"高频任务伤害更大"需要 LEVIR-CD 变化检测对比（阶段 A），'
      '本 notebook 仅给出 EuroSAT 的 MTF 响应曲线，跨任务对比留待阶段 A 补齐。')

## 结果回填

跑完后从左侧 **Output** 面板下载到本地仓库 `eo-degrade/results/`：
- `cloud_scan.csv` / `gsd_scan.csv` / `snr_scan.csv` / `mtf_scan.csv`
- `m2_four_curves.png`（README 主图）

回填后我再做：H1–H4 结论整理 → README 故事化 → git 提交推送。

**口径**：SNR 为功率比 10·log10(m²/(a·m+b))；云遮挡为分位数锁定的浓云覆盖率；
全部固定 SEED=0，可复现。